# EDA on Online Retail Sales

**Oasis Infobyte — Data Analytics Level 1, Task 1**

This notebook analyses the supplied Online Retail dataset.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Initial inspection

In [ ]:
print('Shape:',df_raw.shape)
display(df_raw.dtypes.rename('dtype').to_frame())
display(df_raw.isna().sum().to_frame('missing'))

## 2. Data cleaning

Convert dates, trim text, remove exact duplicates, calculate Revenue, identify cancelled invoices, and exclude cancellations.

In [ ]:
df=df_raw.copy(); before={'rows':len(df),'duplicates':int(df.duplicated().sum()),'missing_cells':int(df.isna().sum().sum())}; df=df.drop_duplicates(); df.columns=df.columns.str.strip(); sales_df=df[(df.InvoiceNo.str[0]!='C') & (df.Quantity>0) & (df.UnitPrice>0)].copy(); sales_df['InvoiceDate']=pd.to_datetime(sales_df['InvoiceDate']); sales_df['Revenue']=sales_df['Quantity']*sales_df['UnitPrice']; after={'rows':len(sales_df),'duplicates':int(sales_df.duplicated().sum()),'missing_cells':int(sales_df.isna().sum().sum())}; print(f"Before: {before}\nAfter: {after}")

## 3. Descriptive statistics

In [ ]:
display(sales_df[['Quantity','UnitPrice','Revenue']].describe().T)
print('Unique invoices:',sales_df.InvoiceNo.nunique())
print('Unique products:',sales_df.Description.nunique())
print('Countries:',sales_df.Country.nunique())

## 4. Monthly and quarterly sales trends

In [ ]:
sales_df['Month']=sales_df.InvoiceDate.dt.to_period('M').astype(str); sales_df['Quarter']=sales_df.InvoiceDate.dt.to_period('Q').astype(str); monthly_revenue=sales_df.groupby('Month').Revenue.sum(); quarterly_revenue=sales_df.groupby('Quarter').Revenue.sum(); print('Peak month:',monthly_revenue.idxmax(),'\nPeak quarter:',quarterly_revenue.idxmax())

### Key visualisations

![Monthly Revenue Trend](outputs/monthly_revenue_trend.png)

![Quarterly Revenue Trend](outputs/quarterly_revenue_trend.png)

## 5. Market analysis

Age and gender are absent from the supplied data. Country analysis is therefore used as the available market/customer dimension.

In [ ]:
country_revenue=sales_df.groupby('Country').Revenue.sum().sort_values(ascending=False); country_orders=sales_df.groupby('Country').InvoiceNo.nunique().sort_values(ascending=False); print('Top 5 countries by revenue:\n',country_revenue.head()); print('\nTop 5 countries by order count:\n',country_orders.head())

### Country analysis visualisations

![Top 10 Countries by Revenue](outputs/top_10_countries_by_revenue.jpg)

![Top 10 Countries by Number of Orders](outputs/top_10_countries_by_orders.jpg)

## 6. Product analysis

In [ ]:
top_products=sales_df.groupby('Description').Quantity.sum().sort_values(ascending=False).head(10); display(top_products)

### Top 10 products by units sold

![Top 10 Products by Units Sold](outputs/top_10_products_by_units_sold.jpg)

## 7. Correlation heatmap

In [ ]:
plt.figure(figsize=(8,6)); sns.heatmap(sales_df[['Quantity','UnitPrice','Revenue']].corr(),annot=True,fmt='.2f',cmap='coolwarm'); plt.title('Correlation Matrix'); plt.tight_layout(); plt.savefig('outputs/correlation_heatmap.png',dpi=300,bbox_inches='tight'); plt.show()

### Correlation matrix

![Correlation Heatmap](outputs/correlation_heatmap.png)

## 8. Additional insight and recommendations

Revenue concentration is assessed by country, alongside customer-level revenue where CustomerID is provided.

In [ ]:
total=sales_df.Revenue.sum(); uk_share=sales_df.loc[sales_df.Country.eq('United Kingdom'),'Revenue'].sum()/total; print(f'UK revenue share: {uk_share:.1%}'); customer_revenue=sales_df.groupby('CustomerID').Revenue.sum().sort_values(ascending=False); print(f'\nTop 5 customers by revenue:\n{customer_revenue.head()}')

## 9. Conclusion

The cleaned dataset supports transaction-level retail sales analysis. The main decision areas are temporal demand planning, geographic market prioritization, and product performance tracking.